# Umbrales por clase de ECGFounder, derivados sobre PTB-XL

Qué hace este notebook, en orden:

1. Baja PTB-XL 1.0.3 (1,7 GB, PhysioNet, CC BY 4.0) y las etiquetas de 150 clases que
   publica ECGFounder para ese dataset (`csv/ptbxl_label.csv`, ya mapeadas).
2. Baja los dos checkpoints de ECGFounder (370 MB cada uno) y clona `ecg-pipeline` en `v0.1.0`.
3. Corre el modelo de 12 derivaciones sobre los 21.799 registros, con y sin el filtro
   pasabanda que upstream usa en fine-tuning (no en su evaluación).
4. Corre el modelo de 1 derivación sobre las derivaciones I, II, V1 y V5 por separado.
   El checkpoint está pensado para I; el pipeline lo usa sobre II/V1/V5, y esto mide cuánto importa.
5. Deriva el umbral por clase con el método de upstream (`util.find_optimal_threshold`:
   barrido 0,01 a 0,99, máximo de balanced accuracy) y guarda:
   `thresholds_12lead.json`, `thresholds_1lead_<lead>.json`, y `metrics.csv` con AUROC,
   positivos y umbral por clase y por variante.

Tiempo en Colab con GPU T4: unos 25 a 40 minutos en total, la mayor parte en bajar y leer los
archivos. En CPU no lo intentes.

Antes de ejecutar: `Entorno de ejecución > Cambiar tipo de entorno > T4 GPU`.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "SIN GPU: cambia el entorno a T4 antes de seguir")
%pip install -q wfdb

## 1. Código y pesos

In [ ]:
%%bash
set -e
cd /content
[ -d ecg-pipeline ] || git clone -q --branch v0.1.1 --depth 1 https://github.com/reeenatamc/ecg-pipeline.git
mkdir -p ecg-pipeline/weights
cd ecg-pipeline/weights
for f in 1_lead_ECGFounder.pth 12_lead_ECGFounder.pth; do
  [ -f "$f" ] || curl -sfL -o "$f" "https://huggingface.co/PKUDigitalHealth/ECGFounder/resolve/main/$f"
done
ls -la

## 2. PTB-XL y las etiquetas de 150 clases

In [ ]:
%%bash
set -e
cd /content
ZIP=ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
[ -d ptbxl ] || {
  [ -f "$ZIP" ] || wget -q --show-progress -O "$ZIP" "https://physionet.org/static/published-projects/ptb-xl/$ZIP"
  unzip -q "$ZIP" && mv ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 ptbxl && rm "$ZIP"
}
[ -f ptbxl_label.csv ] || wget -q -O ptbxl_label.csv https://raw.githubusercontent.com/PKUDigitalHealth/ECGFounder/master/csv/ptbxl_label.csv
ls ptbxl | head; wc -l ptbxl_label.csv

## 3. Carga de datos, exactamente como upstream

In [ ]:
import json, os, sys, time
import numpy as np
import pandas as pd
import torch
import wfdb
from scipy.signal import butter, filtfilt, iirnotch
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, "/content/ecg-pipeline")
from ecg_pipeline.interpret.interpret_ecg import build_12lead_model, build_1lead_model, load_tasks

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PTBXL = "/content/ptbxl/"
TASKS = load_tasks("/content/ecg-pipeline/ecg_pipeline/interpret/tasks.txt")
assert len(TASKS) == 150

# Orden de derivaciones en los archivos WFDB de PTB-XL. Coincide con el canonico del pipeline.
PTBXL_LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

labels = pd.read_csv("/content/ptbxl_label.csv").dropna(subset=["filename_hr", "label"]).reset_index(drop=True)
Y = np.stack([np.asarray(json.loads(l), dtype=np.float32) for l in labels["label"]])
assert Y.shape == (len(labels), 150), Y.shape
print(len(labels), "registros;", int((Y.sum(0) > 0).sum()), "de 150 clases con al menos un positivo en PTB-XL")


def zscore(x):
    # ptbxl_eval.py: global sobre todo el array, igual que waveform.zscore en el pipeline.
    return (x - np.mean(x)) / (np.std(x) + 1e-8)


def filter_bandpass(x, fs=500):
    # util.filter_bandpass de ECGFounder, usado por dataset.py (fine-tuning), no por ptbxl_eval.py.
    b, a = iirnotch(50, 30, fs)
    x = np.stack([filtfilt(b, a, ch) for ch in x])
    b, a = butter(N=4, Wn=[0.67, 40], btype="bandpass", fs=fs)
    return np.stack([filtfilt(b, a, ch) for ch in x])


class PTBXLDataset(Dataset):
    def __init__(self, lead_indices, bandpass):
        self.lead_indices, self.bandpass = lead_indices, bandpass

    def __len__(self):
        return len(labels)

    def __getitem__(self, i):
        sig, _ = wfdb.rdsamp(PTBXL + labels.loc[i, "filename_hr"])
        x = np.nan_to_num(np.transpose(sig, (1, 0)), nan=0.0)[self.lead_indices]  # (C, 5000)
        if self.bandpass:
            x = filter_bandpass(x)
        return torch.from_numpy(np.ascontiguousarray(zscore(x), dtype=np.float32))


@torch.no_grad()
def predict_all(model, lead_indices, bandpass, batch_size=64):
    loader = DataLoader(PTBXLDataset(lead_indices, bandpass), batch_size=batch_size, num_workers=os.cpu_count(), shuffle=False)
    out, t0 = [], time.time()
    for k, x in enumerate(loader):
        out.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
        if k % 50 == 0:
            print(f"  lote {k}/{len(loader)}  {time.time() - t0:.0f}s", flush=True)
    return np.concatenate(out)

## 4. Inferencia: 12 derivaciones (sin y con filtro) y 1 derivación (I, II, V1, V5)

In [ ]:
RUNS = {}  # nombre -> (probs, descripcion)

model12 = build_12lead_model("/content/ecg-pipeline/weights/12_lead_ECGFounder.pth", DEVICE)
all12 = list(range(12))
print("12lead, sin filtro (el camino de ptbxl_eval.py)")
RUNS["12lead"] = (predict_all(model12, all12, bandpass=False), "12 derivaciones, z-score, sin filtro")
print("12lead, con el pasabanda de dataset.py")
RUNS["12lead_bandpass"] = (predict_all(model12, all12, bandpass=True), "12 derivaciones, notch 50 + 0.67-40 Hz, z-score")
del model12; torch.cuda.empty_cache()

model1 = build_1lead_model("/content/ecg-pipeline/weights/1_lead_ECGFounder.pth", DEVICE)
for lead in ["I", "II", "V1", "V5"]:
    print(f"1lead sobre {lead}")
    RUNS[f"1lead_{lead}"] = (predict_all(model1, [PTBXL_LEADS.index(lead)], bandpass=False), f"1 derivacion ({lead}), z-score, sin filtro")
print("1lead sobre II con pasabanda")
RUNS["1lead_II_bandpass"] = (predict_all(model1, [PTBXL_LEADS.index("II")], bandpass=True), "1 derivacion (II), con pasabanda")
del model1; torch.cuda.empty_cache()

os.makedirs("/content/out", exist_ok=True)
np.savez_compressed("/content/out/probs.npz", Y=Y, **{k: v[0] for k, v in RUNS.items()})
print("guardado /content/out/probs.npz")

## 5. Umbrales con el método de upstream, y AUROC por clase

In [ ]:
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

# Por debajo de esto no se emite umbral: un barrido de balanced accuracy sobre tres
# positivos es ruido, y en el pipeline una clase sin umbral se muestra como ranking.
MIN_POSITIVES = 20


def optimal_threshold(gt, pred):
    # util.find_optimal_threshold de ECGFounder, literal.
    best_ba, best = -1.0, 0.5
    for thr in np.linspace(0.01, 0.99, 99):
        ba = balanced_accuracy_score(gt, (pred > thr).astype(int))
        if ba > best_ba:
            best_ba, best = ba, thr
    return float(best), float(best_ba)


def evaluate(name, probs):
    rows, thresholds = [], {}
    for i, task in enumerate(TASKS):
        gt = Y[:, i]
        n_pos = int(gt.sum())
        if n_pos < MIN_POSITIVES or n_pos == len(gt):
            rows.append({"variant": name, "label": task, "positives": n_pos, "auroc": None, "threshold": None, "balanced_acc": None})
            continue
        thr, ba = optimal_threshold(gt, probs[:, i])
        rows.append({"variant": name, "label": task, "positives": n_pos, "auroc": round(float(roc_auc_score(gt, probs[:, i])), 4), "threshold": round(thr, 2), "balanced_acc": round(ba, 4)})
        thresholds[task] = round(thr, 2)
    return rows, thresholds


all_rows = []
for name, (probs, _) in RUNS.items():
    rows, thresholds = evaluate(name, probs)
    all_rows += rows
    with open(f"/content/out/thresholds_{name}.json", "w") as fh:
        json.dump(thresholds, fh, indent=2, sort_keys=True)
    print(f"{name}: {len(thresholds)} clases con umbral")

metrics = pd.DataFrame(all_rows)
metrics.to_csv("/content/out/metrics.csv", index=False)

## 6. Lo que hay que mirar

In [ ]:
# AUROC medio sobre las clases con umbral, por variante. Con y sin filtro responde el punto
# abierto del NOTICE; I vs II/V1/V5 responde si el pipeline puede seguir usando el
# checkpoint de 1 derivacion sobre tiras de ritmo que no son la derivacion I.
scored = metrics.dropna(subset=["auroc"])
summary = scored.groupby("variant").agg(classes=("label", "count"), auroc_mean=("auroc", "mean"), auroc_median=("auroc", "median")).round(4)
summary["descripcion"] = [RUNS[v][1] for v in summary.index]
print(summary.to_string())

KEY = ["SINUS RHYTHM", "NORMAL SINUS RHYTHM", "ATRIAL FIBRILLATION", "SINUS TACHYCARDIA", "SINUS BRADYCARDIA", "NORMAL ECG", "ABNORMAL ECG", "RIGHT BUNDLE BRANCH BLOCK", "LEFT BUNDLE BRANCH BLOCK", "ATRIAL FLUTTER", "1ST DEGREE AV BLOCK", "PREMATURE VENTRICULAR COMPLEXES"]
pivot = scored[scored["label"].isin(KEY)].pivot(index="label", columns="variant", values="auroc")
print("\nAUROC en las clases que el worker enseña mas a menudo:")
print(pivot.round(3).to_string())

print("\nUmbrales de 12lead para esas clases (lo que iria en --thresholds):")
print(scored[(scored["variant"] == "12lead") & scored["label"].isin(KEY)][["label", "positives", "threshold", "auroc"]].to_string(index=False))

## 7. Descargar

Los JSON van a `ecg-pipeline/configs/` y se pasan con `--thresholds`; `metrics.csv` es la
tabla para el README y para la reunión con el cardiólogo. Cómo leerla:

- Si `12lead_bandpass` no supera a `12lead` en AUROC medio, el filtro no se implementa y el
  punto del NOTICE se cierra con ese número.
- Si `1lead_II` queda cerca de `1lead_I` en las clases de ritmo, el pipeline puede seguir
  usando el checkpoint sobre tiras II/V1/V5. Si `1lead_V1` cae mucho, conviene quitar V1 de
  `PREFERRED_RHYTHM_LEADS` o ponderarla menos.
- Los umbrales son de PTB-XL (ECG de máquina, 500 Hz). Sobre un ECG digitalizado de papel
  pueden correrse; eso lo resuelve el set de validación propio, no este notebook.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/ecgfounder_thresholds", "zip", "/content/out")
files.download("/content/ecgfounder_thresholds.zip")